# StartupAI: Machine Learning Model Exploration & Benchmarking

**Tagline:** *Know your startup's potential before you scale.*

This notebook covers the complete data analysis and ML development pipeline for the **StartupAI** platform:
1. Dataset Ingestion & Exploratory Data Analysis (EDA)
2. Data Preprocessing & Column Transformations (`ColumnTransformer`, `OneHotEncoder`, `StandardScaler`)
3. **Model 1 (Primary Success Classifier):** Decision Tree Classifier with Feature Importance Extraction
4. **Model 2 (Secondary Classifier):** K-Nearest Neighbors (k-NN) with Cross-Validation Optimization
5. **Model 3 (Revenue Projection):** Linear Regression with MAE, RMSE, and $R^2$ Evaluation
6. **Model 4 (Startup Segmentation):** K-Means Clustering with Elbow Method & Silhouette Score Analysis

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score, classification_report
)

sns.set_theme(style="whitegrid")
print("Libraries successfully imported.")

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
data_path = '../backend/data/startups.csv'
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
df.describe().T

## 2. Feature Preprocessing Pipeline

In [ ]:
cat_features = ['industry', 'location']
num_features = [
    'startup_age', 'founders', 'employees', 'founder_experience',
    'technical_team_size', 'business_team_size', 'funding', 'investors',
    'funding_rounds', 'monthly_revenue', 'monthly_burn_rate', 'marketing_budget',
    'market_size', 'customers', 'customer_growth', 'customer_retention',
    'customer_acquisition_cost'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

X = df[num_features + cat_features]
y_cls = df['success']
y_rev = df['annual_revenue'] if 'annual_revenue' in df.columns else df['monthly_revenue'] * 12

X_processed = preprocessor.fit_transform(X)
print(f"Processed Feature Space Shape: {X_processed.shape}")

## 3. Algorithm A: Decision Tree Classifier (Primary Success Model)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

dt_model = DecisionTreeClassifier(max_depth=8, min_samples_split=12, min_samples_leaf=6, random_state=42)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)
print("Decision Tree Evaluation:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_dt):.4f}")

## 4. Algorithm B: K-Nearest Neighbors Classifier (Validation Ensemble)

In [ ]:
knn_scaler = StandardScaler()
X_knn_scaled = knn_scaler.fit_transform(df[num_features])
X_train_knn, X_test_knn, y_train_knn, y_test_knn = train_test_split(X_knn_scaled, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Search optimal k
k_values = range(3, 23, 2)
f1_scores = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    score = cross_val_score(knn, X_train_knn, y_train_knn, cv=5, scoring='f1').mean()
    f1_scores.append(score)

best_k = k_values[np.argmax(f1_scores)]
print(f"Optimal k selected: {best_k}")

knn_final = KNeighborsClassifier(n_neighbors=best_k, weights='distance')
knn_final.fit(X_train_knn, y_train_knn)
y_pred_knn = knn_final.predict(X_test_knn)

print(f"k-NN (k={best_k}) Accuracy: {accuracy_score(y_test_knn, y_pred_knn):.4f}")
print(f"k-NN (k={best_k}) F1 Score: {f1_score(y_test_knn, y_pred_knn):.4f}")

## 5. Algorithm C: Linear Regression (Revenue Forecast Engine)

In [ ]:
X_train_rev, X_test_rev, y_train_rev, y_test_rev = train_test_split(X_processed, y_rev, test_size=0.2, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train_rev, y_train_rev)
y_pred_rev = lr_model.predict(X_test_rev)

print("Linear Regression Metrics:")
print(f"MAE:  INR {mean_absolute_error(y_test_rev, y_pred_rev):,.2f}")
print(f"RMSE: INR {np.sqrt(mean_squared_error(y_test_rev, y_pred_rev)):,.2f}")
print(f"R²:   {r2_score(y_test_rev, y_pred_rev):.4f}")

## 6. Algorithm D: K-Means Clustering (Startup Segmentation)

In [ ]:
kmeans_features = ['funding', 'monthly_revenue', 'monthly_burn_rate', 'customer_growth', 'market_size', 'employees', 'founder_experience', 'customers']
km_scaler = StandardScaler()
X_km_scaled = km_scaler.fit_transform(df[kmeans_features])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_km_scaled)

sil = silhouette_score(X_km_scaled, clusters)
print(f"K-Means Silhouette Score: {sil:.4f}")
print(f"Inertia: {kmeans.inertia_:.2f}")